# Escalation Ladder Figure (Section 3.3)

Generates the color-coded escalation ladder figure for the Methodology section.
Tier names, descriptions, groups, and colors all come from `config/paper_labels.yaml`.

**Output:** `latex/figures/escalation_ladder.pdf`

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

ROOT = Path(os.environ.get('HIREWATCH_ROOT', '.')).resolve()
OUT = ROOT / 'outputs' / 'figures' / 'escalation_ladder.pdf'
OUT.parent.mkdir(parents=True, exist_ok=True)

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import importlib
import analysis.labels as _lab
importlib.reload(_lab)
from analysis.labels import TIER_LADDER, TIER_META, LADDER_COLORS

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Nimbus Roman', 'DejaVu Serif'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})
import os


In [ ]:
# Build tier data from YAML config
tiers = []
for _, slug, short, full in TIER_LADDER:
    m = TIER_META[slug]
    tiers.append({
        'num': m['number'],
        'name': m.get('full', short),
        'desc': m.get('description', ''),
        'group': m.get('group', 'context'),
    })

print('Tiers from config/paper_labels.yaml:')
for t in tiers:
    print(f'  {t["num"]}. [{t["group"]:>8}] {t["name"]:<24} {t["desc"]}')

In [ ]:
# ---- Draw the escalation ladder figure ----

def hex_to_rgb(h):
    return tuple(int(h[i:i+2], 16) / 255 for i in (0, 2, 4))

ctx_colors = LADDER_COLORS['context']
prs_colors = LADDER_COLORS['pressure']

def get_colors(group, idx):
    spec = ctx_colors if group == 'context' else prs_colors
    bg_hex = spec['bg'][min(idx, len(spec['bg'])-1)]
    fg_hex = spec['fg'][min(idx, len(spec['fg'])-1)]
    return hex_to_rgb(bg_hex), hex_to_rgb(fg_hex)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.set_xlim(-0.3, 10.5)
ax.set_ylim(-0.5, 7.8)
ax.axis('off')

row_height = 0.85
gap = 0.12
y_start = 7.0

# Column positions
x_num = 0.0
x_name = 0.7
x_desc = 3.8
x_badge = 9.2
badge_w = 1.3

# Header
ax.text(x_name, y_start + 0.6, 'Tier', fontsize=9, fontweight='bold')
ax.text(x_desc, y_start + 0.6, 'Prompt content', fontsize=9, fontweight='bold')
ax.text(x_badge + badge_w/2, y_start + 0.6, 'Condition type', fontsize=8, fontweight='bold', ha='center')
ax.plot([-0.1, 10.3], [y_start + 0.35, y_start + 0.35], color='#999999', lw=0.6)

group_rows = {'context': [], 'pressure': []}
ctx_idx = 0
prs_idx = 0

for i, t in enumerate(tiers):
    y = y_start - i * (row_height + gap)
    group = t['group']

    if group == 'context':
        bg, fg = get_colors('context', ctx_idx)
        ctx_idx += 1
    else:
        bg, fg = get_colors('pressure', prs_idx)
        prs_idx += 1

    group_rows[group].append(y)

    # Pastel row background (columns 1-3 only)
    bg_pastel = tuple(c * 0.3 + 0.7 for c in bg)
    row_rect = FancyBboxPatch(
        (-0.15, y - row_height/2 + 0.05), 8.8, row_height - 0.1,
        boxstyle="round,pad=0.05", facecolor=bg_pastel, edgecolor='none', alpha=0.7
    )
    ax.add_patch(row_rect)

    # Tier number badge
    badge_rect = FancyBboxPatch(
        (x_num, y - 0.22), 0.5, 0.44,
        boxstyle="round,pad=0.08", facecolor=bg, edgecolor='none'
    )
    ax.add_patch(badge_rect)
    ax.text(x_num + 0.25, y, str(t['num']), fontsize=10, fontweight='bold',
            color=fg, ha='center', va='center')

    # Tier name
    ax.text(x_name, y, t['name'], fontsize=9, fontweight='bold', color=fg, va='center')

    # Description
    ax.text(x_desc, y, t['desc'], fontsize=8, color=fg, va='center')

# Dashed divider between groups
div_y = (group_rows['context'][-1] + group_rows['pressure'][0]) / 2
ax.plot([-0.1, 10.3], [div_y, div_y], color='#aaaaaa', lw=0.8, linestyle='--')

# Group badges on the right
for group_key, label, badge_bg_hex, badge_fg_hex in [
    ('context', 'Occupational\ndetails', 'B5D4F4', '0C447C'),
    ('pressure', 'Organizational\npressure', 'F0997B', '712B13'),
]:
    rows = group_rows[group_key]
    y_center = (rows[0] + rows[-1]) / 2
    h = rows[0] - rows[-1] + row_height * 0.8
    badge = FancyBboxPatch(
        (x_badge, y_center - h/2), badge_w, h,
        boxstyle="round,pad=0.12", facecolor=hex_to_rgb(badge_bg_hex), edgecolor='none'
    )
    ax.add_patch(badge)
    ax.text(x_badge + badge_w/2, y_center, label,
            fontsize=8, fontweight='bold', color=hex_to_rgb(badge_fg_hex),
            ha='center', va='center', linespacing=1.4)

# Bottom line
ax.plot([-0.1, 10.3], [y_start - len(tiers) * (row_height + gap) + gap + 0.35,
                         y_start - len(tiers) * (row_height + gap) + gap + 0.35],
        color='#999999', lw=0.6)

plt.tight_layout()
fig.savefig(OUT, bbox_inches='tight')
print(f'Saved: {OUT}')
plt.show()